<a href="https://colab.research.google.com/github/douglaskorvo/tourism_supply_chain/blob/main/tssc_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# Tourism service supply chains — audited reproducible analysis

**Version 1.1.0.** This notebook implements the complete computational pipeline used for the reported analysis: cleaning and exclusions, frozen dictionary coding, descriptive tables, supplier-clustered logistic models, Firth rare-events checks, FDR correction, formal equivalence tests, subsample robustness, leave-one-out same-source assessment, country heterogeneity and publication figures.

The raw review corpus is not distributed. Set `TSSC_RAW_CSV` to a lawfully obtained export with the documented schema. The public repository contains the computational method and aggregate reference outputs; individual human-coding records and validation metrics are outside the scope of this public version, so the notebook neither reconstructs nor invents them.

Douglas De Souza Rodrigues — Production Engineering Department, Fluminense Federal University  
ORCID: 0000-0001-7473-7425


## 1. Environment and configuration


In [ ]:
import hashlib, json, os, platform, re, sys, time, unicodedata, warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy import stats
from scipy.special import expit
from scipy.stats import chi2, norm
import statsmodels
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170, "display.max_columns", 50)

STARTED = time.time()
CFG = {
    "raw_csv": Path(os.getenv("TSSC_RAW_CSV", "reviews_raw.csv")),
    "outdir": Path(os.getenv("TSSC_OUTDIR", "output")),
    "reference_dir": Path(os.getenv("TSSC_REFERENCE_DIR", "reference_outputs")),
    "seed": 20260817,
    "fe_main": "C(city)",
    "fe_sensitivity": "C(country)",
    "optimizer": "bfgs",
    "maxiter": 500,
    "cluster": "place_id",
    "min_country_events": 15,
    "reference_raw_sha256": "a4d34c3b3f89be548c6345f1a68370674441c7110431498996f1845714746fa0",
}
CFG["outdir"].mkdir(parents=True, exist_ok=True)
np.random.seed(CFG["seed"])

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 0.8, "figure.dpi": 150,
    "savefig.dpi": 300, "savefig.bbox": "tight",
})
C_SHORT, C_LONG, C_NEUT = "#2E5E8C", "#C1663A", "#555555"

print(f"Python {platform.python_version()} ({sys.platform})")
for module in (pd, np, statsmodels, scipy, matplotlib):
    print(f"  {module.__name__:12s} {module.__version__}")
print("Raw input:", CFG["raw_csv"])
print("Output directory:", CFG["outdir"])


## 2. Data loading, exclusions and derived variables


In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

raw_sha256 = sha256_file(CFG["raw_csv"])
REFERENCE_RUN = raw_sha256 == CFG["reference_raw_sha256"]
print("Raw SHA-256:", raw_sha256)
print("Reference corpus:", REFERENCE_RUN)

LOG = []
def log_step(label, n):
    LOG.append({"step": label, "records": int(n)})
    print(f"{label:<68s} n = {n:>6,}")

raw = pd.read_csv(CFG["raw_csv"])
required = {"place_id", "country", "city", "segment", "author_name", "rating", "text", "time_desc", "lang"}
missing = required.difference(raw.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

log_step("0. Raw export", len(raw))
df = raw.copy()
for col in ["place_id", "country", "city", "segment", "author_name", "time_desc", "lang"]:
    df[col] = df[col].astype(str).str.strip()

NON_LEXICAL_SYMBOLS = "".join(chr(codepoint) for codepoint in [
    0x1FAF6, 0x1FA77, 0x1F979, 0x1FA75, 0x1FAF0, 0x1FAE1, 0x1FA76,
    0x1FAB7, 0x1FAE0, 0x1FABB, 0x1FAF4, 0x1FAE3, 0x1FAE4, 0x1FAF1,
    0x1FAF2, 0x1F6DC, 0x1FAA9, 0x1FAE2, 0x1FAB8,
])
NON_LEXICAL_PATTERN = "[" + re.escape(NON_LEXICAL_SYMBOLS) + "]"

def clean_text(value):
    """Unicode NFC, removal of delivery labels/control characters and whitespace normalisation."""
    if not isinstance(value, str):
        return np.nan
    value = unicodedata.normalize("NFC", value)
    value = re.sub(r"\(Traduzido pelo Google\)|\(Original\)", " ", value)
    value = "".join(ch for ch in value if ch == "\n" or unicodedata.category(ch)[0] != "C")
    # Symbols observed as standalone whitespace tokens in the frozen export;
    # removing them makes lexical length deterministic without changing dictionary matches.
    value = re.sub(NON_LEXICAL_PATTERN, "", value)
    value = re.sub(r"[ \t ]+", " ", value)
    value = re.sub(r"\n{3,}", "\n\n", value).strip()
    return value or np.nan

df["text"] = df["text"].apply(clean_text)
ambiguous_ids = set(df.groupby("place_id")["segment"].nunique().loc[lambda s: s > 1].index)

df = df[df["text"].notna()].copy()
log_step("1. After removing records without text", len(df))

before = len(df)
df = df.drop_duplicates(subset=["place_id", "author_name", "text"], keep="first").copy()
log_step(f"2. After removing exact duplicates (-{before - len(df)})", len(df))

df["repeated_text"] = df.duplicated(subset=["text"], keep=False) & df["text"].str.len().gt(15)
df["ambiguous_config"] = df["place_id"].isin(ambiguous_ids)
df["segment"] = df["place_id"].map(
    df.groupby("place_id")["segment"].agg(lambda s: s.value_counts().index[0])
)
df = df[~df["ambiguous_config"]].copy()
log_step(f"3. After excluding {len(ambiguous_ids)} ambiguous-configuration providers", len(df))

T0 = pd.DataFrame(LOG)
T0.to_csv(CFG["outdir"] / "T0_exclusion_log.csv", index=False)

df["negative"] = df["rating"].le(2).astype(int)
df["n_words"] = df["text"].str.split().str.len()
df["log_words"] = np.log1p(df["n_words"])

PT_NUM = {"um": 1, "uma": 1, "dois": 2, "duas": 2}
def to_years(value):
    value = value.lower()
    if any(token in value for token in ("hora", "dia", "última semana", "ultima semana")):
        return 0.0
    match = re.search(r"(\d+)", value)
    number = int(match.group(1)) if match else PT_NUM.get(value.split()[0], 1)
    if "semana" in value:
        return round(number * 7 / 365, 3)
    if "mês" in value or "mes" in value:
        return round(number / 12, 3)
    if "ano" in value:
        return float(number)
    return np.nan

df["review_age"] = df["time_desc"].apply(to_years)
df["translated"] = np.where(
    df["country"].eq("Brazil"),
    df["lang"].eq("pt-BR").astype(int),
    (~df["lang"].isin(["pt"])).astype(int),
)
df["mass_market"] = df["segment"].eq("mass_market").astype(int)

print(f"Analytical corpus: {len(df):,} reviews; {df.place_id.nunique():,} providers; "
      f"{df.city.nunique()} cities; {df.country.nunique()} countries")


## 3. Frozen dictionaries and rule-based coding


In [ ]:
# ------------------------------------------------------------ frozen dictionaries
STAGES = {
 "stage_booking"   : r"reserv(a|ei|amos|ado)|agend(ei|amos|ado)|booking|pagamento|paguei|cart[ãa]o de cr[ée]dito|dep[óo]sito|contrat(ei|amos|ado)",
 "stage_transport" : r"motorista|[ôo]nibus|van\b|traslado|transfer\b|micro-?[ôo]nibus|ve[íi]culo|aeroporto|nos buscou|buscar no hotel|jipe|4x4|lancha|barco",
 "stage_lodging"   : r"hotel|pousada|hospedagem|resort|hostel|acomoda[çc][ãa]o|di[áa]rias",
 "stage_food"      : r"almo[çc]o|jantar|caf[ée] da manh[ãa]|refei[çc][ãa]o|restaurante|lanche|degusta[çc][ãa]o|comida servida",
 "stage_delivery"  : r"guia|instrutor|professor|monitor|anfitri[ãa]o|oficina|aula|passeio|tour\b|atividade|experi[êe]ncia",
}

CONSTRUCTS = {
 "intermediation" : r"ag[êe]nci|operadora|tour operator|travels?\b|intermedi|terceiriz|subcontrat|revend|pacote (tur[íi]stico|fechado|completo)",
 "coordination"   : r"bem organizad|super organizad|tudo organizad|organiza[çc][ãa]o (foi|impec|perfeita|excelente)|planejad|itiner[áa]rio|roteiro|coordena[çc]|log[íi]stic|bem estruturad|correu tudo (bem|certo)|sem contratempo|pontualidade",
 "delay"          : r"atras(o|os|ou|ado|ada|aram)|demor(ou|ada|ado|amos)|tempo de espera|longa espera|horas? de espera|fila (enorme|imensa|gigante)|esperamos (mais de|por|quase)|aguardamos (mais de|por)|n[ãa]o chegou no hor[áa]rio",
 "digital"        : r"whatsapp|site\b|website|aplicativo|\bapp\b|on-?line|e-?mail|instagram|plataforma|link\b|reserva pela internet|formul[áa]rio",
 "environmental"  : r"meio ambiente|ambiental|sustent[áa]vel|sustentabilidade|ecol[óo]gic|reciclag|lixo|polui[çc]|impacto ambiental|pegada de carbono|conserva[çc][ãa]o da natureza|preserva[çc][ãa]o (ambiental|da natureza)|org[âa]nic",
 "labour"         : r"equipe|funcion[áa]ri|colaborador|treinad|capacitad|\bstaff\b|profissionalismo|artes[ãa]|comunidade local|moradores locais|gera[çc][ãa]o de (renda|emprego)",
 "price"          : r"pre[çc]o|caro|barato|taxa (extra|escondida|adicional)|cobra(ram|nça|do)|custo|valor (cobrado|pago)|abusiv|superfaturad|custo-?benef[íi]cio",
}

DISRUPTION = (
 r"cancel(ou|aram|ado|ada|amento)|"
 r"(?:n[ãa]o|nunca|sem)\W+(?:\w+\W+){0,2}?(?:apareceu|apareceram|veio|vieram|compareceu|cumpriu|cumpriram|honrou|entregou|entregaram|devolveu|devolveram|reembolsou|reembolsaram)|"
 r"nos deixou na m[ãa]o|deixaram (?:a gente|n[óo]s) na m[ãa]o|remarcaram sem|sem aviso pr[ée]vio|"
 r"golpe|fraude|enganad|estelionat|n[ãa]o cumpriram o (?:combinado|prometido)")

RECOVERY = (
 r"resolve(u|ram)|resolvid|solucion(ou|aram)|corrigi(u|ram)|pediu desculpa|se desculp|"
 r"reembolsaram|devolveram o dinheiro|compensa(ram|ção)|trocaram por|deram um jeito|"
 r"remarcaram (para|sem custo)|nos acomodaram|refizeram")

# negation / non-disruptive exclusions (Section 2.4)
NO_DELAY   = r"sem (?:nenhum )?(?:atraso|demora)|n[ãa]o (?:houve|teve|tivemos|tiveram) (?:nenhum )?atraso|nenhum atraso|zero atraso"
BENIGN_CXL = r"pol[íi]tica de cancelamento|cancelamento (gr[áa]tis|gratuito|flex[íi]vel|sem custo)|pode(m)? cancelar|cancelei|cancelamos|cancelar com anteced"


In [ ]:
low = df.text.str.lower()
def match(pat):
    return low.str.contains(pat, regex=True, na=False).astype(int)

for k, p in {**STAGES, **CONSTRUCTS}.items():
    df[k] = match(p)

df["disruption"]     = match(DISRUPTION)
df["recovery_terms"] = match(RECOVERY)

# --- negation and non-disruptive-use corrections
df.loc[match(BENIGN_CXL).astype(bool)
       & ~low.str.contains(r"cancel(?:ou|aram)\b", regex=True, na=False), "disruption"] = 0
df.loc[match(NO_DELAY).astype(bool)
       & ~low.str.contains(r"atras(?:o|ou|ado|aram) (?:de|no|na|em) ", regex=True, na=False), "delay"] = 0

# recovery is only meaningful conditional on a reported disruption
df["recovery"]    = ((df.disruption == 1) & (df.recovery_terms == 1)).astype(int)
df["stage_breadth"] = df[list(STAGES)].sum(axis=1)

print(f"reported disruptions: {int(df.disruption.sum()):,} ({df.disruption.mean()*100:.2f}% of reviews)")
print(f"disruption + recovery language: {int(df.recovery.sum())}")

if REFERENCE_RUN:
    assert len(df) == 51037
    assert df.place_id.nunique() == 11500
    assert int(df.disruption.sum()) == 486
    assert int(df.recovery.sum()) == 44


## 4. Descriptive tables


In [ ]:
country_rows = []
for country, group in df.groupby("country"):
    country_rows.append({
        "country": country,
        "cities": group.city.nunique(),
        "providers": group.place_id.nunique(),
        "independent": int((group.mass_market == 0).sum()),
        "intermediated": int((group.mass_market == 1).sum()),
        "reviews": len(group),
        "mean_rating": group.rating.mean(),
        "translated_pct": 100 * group.translated.mean(),
        "negative_pct": 100 * group.negative.mean(),
    })
T1 = pd.DataFrame(country_rows).sort_values("reviews", ascending=False)
T1 = pd.concat([T1, pd.DataFrame([{
    "country": "TOTAL", "cities": df.city.nunique(), "providers": df.place_id.nunique(),
    "independent": int((df.mass_market == 0).sum()),
    "intermediated": int((df.mass_market == 1).sum()), "reviews": len(df),
    "mean_rating": df.rating.mean(), "translated_pct": 100 * df.translated.mean(),
    "negative_pct": 100 * df.negative.mean(),
}])], ignore_index=True).round(4)
T1.to_csv(CFG["outdir"] / "T1_sample_composition.csv", index=False)
display(T1)

construct_order = ["disruption", "delay", "coordination", "intermediation", "digital", "price", "labour", "environmental", "recovery"]
T2 = pd.DataFrame([{
    "construct": name,
    "positive_n": int(df[name].sum()),
    "pct_total": 100 * df[name].mean(),
    "pct_independent": 100 * df.loc[df.mass_market.eq(0), name].mean(),
    "pct_intermediated": 100 * df.loc[df.mass_market.eq(1), name].mean(),
} for name in construct_order]).round(4)
T2.to_csv(CFG["outdir"] / "T2_construct_prevalence.csv", index=False)
display(T2)

T3 = pd.DataFrame([
    {
        "stratum": label,
        "exposure_pct": 100 * group.disruption.mean(),
        "severity_pct": 100 * group.loc[group.disruption.eq(1), "negative"].mean(),
        "net_negative_pct": 100 * group.negative.mean(),
        "disruptions_n": int(group.disruption.sum()),
    }
    for label, group in [
        ("independent", df[df.mass_market.eq(0)]),
        ("intermediated", df[df.mass_market.eq(1)]),
    ]
]).round(4)
T3.to_csv(CFG["outdir"] / "T3_decomposition.csv", index=False)
display(T3)

T12 = (df.groupby("stage_breadth")
       .agg(n=("rating", "size"), disruption_pct=("disruption", lambda s: 100 * s.mean()),
            negative_pct=("negative", lambda s: 100 * s.mean()))
       .reset_index())
T12.to_csv(CFG["outdir"] / "T12_stage_breadth.csv", index=False)
display(T12)


## 5. Estimation helpers


In [ ]:
CTRL = "log_words + review_age + translated"

def fit_logit(formula, data, clustered=True):
    kwargs = {"method": CFG["optimizer"], "maxiter": CFG["maxiter"], "disp": 0}
    if clustered:
        kwargs.update(cov_type="cluster", cov_kwds={"groups": data[CFG["cluster"]]})
    return smf.logit(formula, data=data).fit(**kwargs)

def converged(model):
    return bool(model.mle_retvals.get("converged", True))

def or_record(model, model_name, term):
    ci = model.conf_int().loc[term]
    return {
        "model": model_name, "term": term,
        "OR": np.exp(model.params[term]),
        "CI_low": np.exp(ci.iloc[0]), "CI_high": np.exp(ci.iloc[1]),
        "p": model.pvalues[term], "n": int(model.nobs), "converged": converged(model),
    }

def full_or_table(model, model_name):
    ci = model.conf_int()
    rows = []
    for term in model.params.index:
        if term == "Intercept" or term.startswith("C("):
            continue
        rows.append({
            "model": model_name, "term": term,
            "OR": np.exp(model.params[term]),
            "CI_low": np.exp(ci.loc[term, 0]), "CI_high": np.exp(ci.loc[term, 1]),
            "p": model.pvalues[term], "n": int(model.nobs),
            "clusters": np.nan,
            "pseudo_R2": model.prsquared, "AIC": model.aic, "converged": converged(model),
        })
    return pd.DataFrame(rows)

def bh_adjust(pvalues):
    rejected, qvalues, _, _ = multipletests(np.asarray(pvalues, dtype=float), alpha=0.05, method="fdr_bh")
    return rejected, qvalues

def firth_logit(formula, data, maxiter=200, tol=1e-8):
    """Firth logistic regression via adjusted score iterations and Jeffreys-prior penalisation."""
    model = smf.logit(formula, data=data)
    X = np.asarray(model.exog, dtype=float)
    y = np.asarray(model.endog, dtype=float)
    beta = np.zeros(X.shape[1])

    def penalised_loglik(b):
        probability = np.clip(expit(X @ b), 1e-15, 1 - 1e-15)
        loglik = np.sum(y * np.log(probability) + (1 - y) * np.log(1 - probability))
        information = X.T @ ((probability * (1 - probability))[:, None] * X)
        sign, logdet = np.linalg.slogdet(information)
        return loglik + 0.5 * logdet if sign > 0 else -np.inf

    for iteration in range(maxiter):
        probability = np.clip(expit(X @ beta), 1e-9, 1 - 1e-9)
        weight = probability * (1 - probability)
        information = X.T @ (weight[:, None] * X)
        information_inv = np.linalg.pinv(information)
        leverage = weight * np.einsum("ij,jk,ik->i", X, information_inv, X)
        adjusted_score = X.T @ (y - probability + leverage * (0.5 - probability))
        step = information_inv @ adjusted_score
        scale = 1.0
        old_value = penalised_loglik(beta)
        while scale > 1e-8 and penalised_loglik(beta + scale * step) < old_value:
            scale /= 2
        updated = beta + scale * step
        if np.max(np.abs(updated - beta)) < tol:
            beta = updated
            break
        beta = updated

    probability = np.clip(expit(X @ beta), 1e-9, 1 - 1e-9)
    information = X.T @ ((probability * (1 - probability))[:, None] * X)
    covariance = np.linalg.pinv(information)
    standard_error = np.sqrt(np.diag(covariance))
    pvalue = 2 * norm.sf(np.abs(beta / standard_error))
    return {
        "names": model.exog_names, "params": beta, "se": standard_error,
        "pvalues": pvalue, "converged": iteration < maxiter - 1, "iterations": iteration + 1,
    }


## 6. Main models and complete coefficient table


In [ ]:
m1_city = fit_logit(f"disruption ~ mass_market + {CTRL} + {CFG['fe_main']}", df)
m1_country = fit_logit(f"disruption ~ mass_market + {CTRL} + {CFG['fe_sensitivity']}", df)
m2_city = fit_logit(f"negative ~ mass_market * disruption + {CTRL} + {CFG['fe_main']}", df)
m_stage_disruption = fit_logit(f"disruption ~ stage_breadth + {CTRL} + {CFG['fe_main']}", df)
m_stage_negative = fit_logit(f"negative ~ stage_breadth + {CTRL} + {CFG['fe_main']}", df)

eq3_constructs = ["intermediation", "coordination", "delay", "digital", "price", "labour", "environmental"]
eq3_formula = (
    "negative ~ mass_market + disruption + " + " + ".join(eq3_constructs)
    + f" + {CTRL} + {CFG['fe_main']}"
)
m3_city = fit_logit(eq3_formula, df)

T4 = pd.DataFrame([
    or_record(m1_city, "Eq1 exposure (city FE)", "mass_market"),
    or_record(m1_country, "Eq1 exposure (country FE)", "mass_market"),
    or_record(m2_city, "Eq2 (city FE)", "mass_market"),
    or_record(m2_city, "Eq2 (city FE)", "disruption"),
    or_record(m2_city, "Eq2 (city FE)", "mass_market:disruption"),
    or_record(m_stage_disruption, "Stage breadth -> disruption", "stage_breadth"),
    or_record(m_stage_negative, "Stage breadth -> negative", "stage_breadth"),
])
T4.to_csv(CFG["outdir"] / "T4_models.csv", index=False)
display(T4)

full_tables = []
for label, model in [
    ("Eq1 exposure (city FE)", m1_city),
    ("Eq1 exposure (country FE)", m1_country),
    ("Eq2 negative rating (city FE)", m2_city),
    ("Eq3 operational profile (city FE)", m3_city),
]:
    table = full_or_table(model, label)
    table["clusters"] = df.place_id.nunique()
    full_tables.append(table)
T4_full = pd.concat(full_tables, ignore_index=True)
T4_full.to_csv(CFG["outdir"] / "T4_full_models.csv", index=False)
display(T4_full)


## 7. Rare-events penalisation and FDR control


In [ ]:
firth_eq1 = firth_logit(f"disruption ~ mass_market + {CTRL} + {CFG['fe_main']}", df)
firth_eq2 = firth_logit(f"negative ~ mass_market * disruption + {CTRL} + {CFG['fe_main']}", df)

def firth_term(result, term):
    index = result["names"].index(term)
    return np.exp(result["params"][index]), result["pvalues"][index]

eq1_firth_or, eq1_firth_p = firth_term(firth_eq1, "mass_market")
eq2_firth_or, eq2_firth_p = firth_term(firth_eq2, "mass_market:disruption")
eq1_ml_or = np.exp(m1_city.params["mass_market"])
eq2_ml_or = np.exp(m2_city.params["mass_market:disruption"])

T5 = pd.DataFrame([
    {
        "model": "Eq1 exposure", "term": "mass_market", "OR_ml": eq1_ml_or,
        "p_ml": m1_city.pvalues["mass_market"], "OR_firth": eq1_firth_or,
        "p_firth": eq1_firth_p,
        "logOR_shift_pct": 100 * abs(np.log(eq1_firth_or) - np.log(eq1_ml_or)) / abs(np.log(eq1_ml_or)),
    },
    {
        "model": "Eq2 interaction", "term": "mass_market:disruption", "OR_ml": eq2_ml_or,
        "p_ml": m2_city.pvalues["mass_market:disruption"], "OR_firth": eq2_firth_or,
        "p_firth": eq2_firth_p,
        "logOR_shift_pct": 100 * abs(np.log(eq2_firth_or) - np.log(eq2_ml_or)) / abs(np.log(eq2_ml_or)),
    },
])
T5.to_csv(CFG["outdir"] / "T5_firth.csv", index=False)
display(T5)

fdr_rows = []
for construct in eq3_constructs:
    fdr_rows.append({
        "construct": construct,
        "OR": np.exp(m3_city.params[construct]),
        "p_raw": m3_city.pvalues[construct],
    })
T6 = pd.DataFrame(fdr_rows)
T6["survives"], T6["q_fdr_bh"] = bh_adjust(T6["p_raw"])
T6 = T6[["construct", "OR", "p_raw", "q_fdr_bh", "survives"]].sort_values("OR", ascending=False)
T6.to_csv(CFG["outdir"] / "T6_fdr_constructs.csv", index=False)
display(T6)


## 8. Average marginal effects and formal equivalence tests


In [ ]:
def marginal_equivalence(fixed_effects, specification):
    model = fit_logit(f"negative ~ mass_market + {CTRL} + {fixed_effects}", df)
    effects = model.get_margeff(at="overall", method="dydx", dummy=False).summary_frame()
    row = effects.loc["mass_market"]
    estimate = float(row.iloc[0])
    standard_error = float(row.iloc[1])
    p_difference = float(row.iloc[3])
    ci_low, ci_high = float(row.iloc[4]), float(row.iloc[5])
    rows = []
    for margin_pp in (0.5, 1.0, 2.0):
        margin = margin_pp / 100
        p_lower = norm.sf((estimate + margin) / standard_error)
        p_upper = norm.cdf((estimate - margin) / standard_error)
        p_tost = max(p_lower, p_upper)
        rows.append({
            "specification": specification,
            "AME_pp": 100 * estimate, "SE_pp": 100 * standard_error,
            "CI95_low_pp": 100 * ci_low, "CI95_high_pp": 100 * ci_high,
            "p_difference": p_difference, "margin_pp": margin_pp,
            "p_TOST": p_tost, "equivalent": p_tost < 0.05,
        })
    return rows

T9 = pd.DataFrame(
    marginal_equivalence(CFG["fe_main"], "city_fe")
    + marginal_equivalence(CFG["fe_sensitivity"], "country_fe")
)
T9.to_csv(CFG["outdir"] / "T9_equivalence_tost.csv", index=False)
display(T9)


## 9. Subsample robustness and leave-one-out same-source checks


In [ ]:
provider_review_count = df.groupby("place_id")["place_id"].transform("size")
subsets = {
    "complete": df,
    "text_ge_50": df[df.n_words.ge(50)],
    "exclude_brazil": df[df.country.ne("Brazil")],
    "translated_only": df[df.translated.eq(1)],
    "exclude_repeated_text": df[~df.repeated_text],
    "reviews_le_2y": df[df.review_age.le(2)],
    "providers_gt_1_review": df[provider_review_count.gt(1)],
}

def robust_term(formula, data, term):
    adjusted = formula
    for variable in ["translated", "review_age"]:
        if data[variable].nunique() < 2:
            adjusted = adjusted.replace(f" + {variable}", "")
    model = fit_logit(adjusted, data)
    return np.exp(model.params[term]), model.pvalues[term], int(model.nobs), converged(model)

robust_rows = []
for label, subset in subsets.items():
    eq1_or, eq1_p, n, eq1_converged = robust_term(
        f"disruption ~ mass_market + {CTRL} + {CFG['fe_main']}", subset, "mass_market"
    )
    eq2_or, eq2_p, _, eq2_converged = robust_term(
        f"negative ~ mass_market * disruption + {CTRL} + {CFG['fe_main']}",
        subset, "mass_market:disruption"
    )
    robust_rows.append({
        "subset": label, "n": n, "eq1_OR": eq1_or, "eq1_p": eq1_p,
        "eq2_interaction_OR": eq2_or, "eq2_interaction_p": eq2_p,
        "eq1_converged": eq1_converged, "eq2_converged": eq2_converged,
    })
T10 = pd.DataFrame(robust_rows)
T10.to_csv(CFG["outdir"] / "T10_robustness.csv", index=False)
display(T10)

grouped = df.groupby("place_id")
df["coord_loo"] = (grouped.coordination.transform("sum") - df.coordination) / (provider_review_count - 1)
df["disr_loo"] = (grouped.disruption.transform("sum") - df.disruption) / (provider_review_count - 1)
multi = df[provider_review_count.gt(1)].copy()
m_loo = fit_logit(f"negative ~ coord_loo + disr_loo + mass_market + {CTRL} + {CFG['fe_main']}", multi)
T11 = pd.DataFrame([or_record(m_loo, "leave-one-out", term) for term in ["coord_loo", "disr_loo", "mass_market"]])
T11["OR_per_10pp"] = np.where(
    T11.term.isin(["coord_loo", "disr_loo"]), np.power(T11.OR, 0.1), T11.OR
)
T11.to_csv(CFG["outdir"] / "T11_leave_one_out.csv", index=False)
display(T11)


## 10. Country heterogeneity and joint tests


In [ ]:
events_by_country = df.groupby("country").disruption.sum()
eligible_countries = events_by_country[events_by_country.ge(CFG["min_country_events"])].index

country_rows = []
for country, subset in df.groupby("country"):
    events = int(subset.disruption.sum())
    if events < CFG["min_country_events"]:
        country_rows.append({
            "country": country, "events": events, "omitted": True,
            "OR": np.nan, "CI_low": np.nan, "CI_high": np.nan,
            "p_raw": np.nan, "q_fdr_bh": np.nan, "survives": np.nan,
        })
        continue
    model = fit_logit("disruption ~ mass_market + log_words + review_age", subset)
    ci = model.conf_int().loc["mass_market"]
    country_rows.append({
        "country": country, "events": events, "omitted": False,
        "OR": np.exp(model.params["mass_market"]),
        "CI_low": np.exp(ci.iloc[0]), "CI_high": np.exp(ci.iloc[1]),
        "p_raw": model.pvalues["mass_market"],
        "q_fdr_bh": np.nan, "survives": False,
    })

T7 = pd.DataFrame(country_rows)
included = ~T7.omitted
rejected, qvalues = bh_adjust(T7.loc[included, "p_raw"])
T7.loc[included, "q_fdr_bh"] = qvalues
T7.loc[included, "survives"] = rejected
T7_included = T7[included].sort_values("OR", ascending=False)
T7_omitted = T7[~included].sort_values("country")
T7 = pd.concat([T7_included, T7_omitted], ignore_index=True)
T7.to_csv(CFG["outdir"] / "T7_countries.csv", index=False)
display(T7)

joint_data = df[df.country.isin(eligible_countries)].copy()
joint_null = f"disruption ~ mass_market + C(country) + {CTRL}"
joint_alt = f"disruption ~ mass_market * C(country) + {CTRL}"
m_joint_null = fit_logit(joint_null, joint_data, clustered=False)
m_joint_conventional = fit_logit(joint_alt, joint_data, clustered=False)
m_joint_clustered = fit_logit(joint_alt, joint_data, clustered=True)

interaction_terms = [name for name in m_joint_conventional.params.index if "mass_market:" in name]
restriction = np.zeros((len(interaction_terms), len(m_joint_conventional.params)))
for row_index, term in enumerate(interaction_terms):
    restriction[row_index, m_joint_conventional.params.index.get_loc(term)] = 1

lr_statistic = 2 * (m_joint_conventional.llf - m_joint_null.llf)
wald_conventional = m_joint_conventional.wald_test(restriction, use_f=True, scalar=True)
wald_clustered = m_joint_clustered.wald_test(restriction, use_f=True, scalar=True)
T8 = pd.DataFrame([
    {"test": "likelihood ratio", "statistic": lr_statistic, "df": len(interaction_terms), "p": chi2.sf(lr_statistic, len(interaction_terms))},
    {"test": "Wald, conventional VCE", "statistic": float(wald_conventional.statistic), "df": len(interaction_terms), "p": float(wald_conventional.pvalue)},
    {"test": "Wald, supplier-clustered VCE", "statistic": float(wald_clustered.statistic), "df": len(interaction_terms), "p": float(wald_clustered.pvalue)},
])
T8.to_csv(CFG["outdir"] / "T8_joint_heterogeneity.csv", index=False)
display(T8)


## 11. Alternative outcome and provider-level summaries


In [ ]:
ols = smf.ols(
    f"rating ~ mass_market + disruption + delay + coordination + {CTRL} + {CFG['fe_main']}", data=df
).fit(cov_type="cluster", cov_kwds={"groups": df.place_id})
print("OLS coefficients on the 1–5 rating scale")
for term in ["mass_market", "disruption", "delay", "coordination"]:
    print(f"  {term:14s} b={ols.params[term]:+.4f}; p={ols.pvalues[term]:.6g}")

provider = (df.groupby("place_id")
            .agg(country=("country", "first"), city=("city", "first"),
                 mass_market=("mass_market", "first"), n=("rating", "size"),
                 mean_rating=("rating", "mean"), disruption_share=("disruption", "mean"),
                 words=("n_words", "mean"))
            .reset_index())
provider["log_words"] = np.log1p(provider.words)
provider_public = provider.copy()
provider_public["place_id"] = "P" + pd.Series(pd.factorize(provider_public.place_id)[0]).astype(str).str.zfill(5)
provider_public.to_csv(CFG["outdir"] / "provider_level_deidentified.csv", index=False)


## 12. Publication figures


In [ ]:
# Figure 1: stage and construct prevalence by collection stratum
plot_vars = list(STAGES) + ["intermediation", "coordination", "delay", "digital", "price", "labour", "environmental"]
labels = [name.replace("stage_", "").replace("_", " ").title() for name in plot_vars]
independent = [100 * df.loc[df.mass_market.eq(0), name].mean() for name in plot_vars]
intermediated = [100 * df.loc[df.mass_market.eq(1), name].mean() for name in plot_vars]
y = np.arange(len(plot_vars)); height = 0.38
fig, ax = plt.subplots(figsize=(7.2, 5.4))
ax.barh(y + height/2, independent, height, color=C_SHORT, label="Independent")
ax.barh(y - height/2, intermediated, height, color=C_LONG, label="Intermediated mass-market")
ax.set_yticks(y, labels); ax.invert_yaxis(); ax.set_xlabel("Reviews mentioning construct (%)")
ax.legend(frameon=False); ax.set_title("Construct prevalence by collection stratum", loc="left", fontweight="bold")
fig.savefig(CFG["outdir"] / "fig1_prevalence_by_stratum.png"); plt.show()

# Figure 2: jointly adjusted operational correlates of a negative rating
forest = T6.sort_values("OR")
ci = m3_city.conf_int()
lo = [np.exp(ci.loc[name, 0]) for name in forest.construct]
hi = [np.exp(ci.loc[name, 1]) for name in forest.construct]
fig, ax = plt.subplots(figsize=(6.2, 3.8)); y = np.arange(len(forest))
ax.hlines(y, lo, hi, color=C_NEUT, lw=1.6)
ax.scatter(forest.OR, y, color=[C_LONG if keep else "#AAAAAA" for keep in forest.survives], zorder=3)
ax.axvline(1, color=C_NEUT, ls="--", lw=0.8); ax.set_xscale("log")
ax.set_yticks(y, [name.replace("_", " ").title() for name in forest.construct])
ax.set_xlabel("Odds ratio for a negative rating (log scale)")
ax.set_title("Jointly adjusted operational correlates", loc="left", fontweight="bold")
fig.savefig(CFG["outdir"] / "fig2_correlates_forest.png"); plt.show()

# Figure 3: exposure, conditional severity and aggregate outcome
panels = [("exposure_pct", "Reported disruption"), ("severity_pct", "Negative | disruption"), ("net_negative_pct", "Negative rating")]
fig, axes = plt.subplots(1, 3, figsize=(7.4, 2.9))
for axis, (column, title) in zip(axes, panels):
    values = T3[column].to_numpy()
    bars = axis.bar(["Independent", "Intermediated"], values, color=[C_SHORT, C_LONG], width=0.62)
    for bar, value in zip(bars, values):
        axis.text(bar.get_x() + bar.get_width()/2, value + max(values)*0.03, f"{value:.2f}", ha="center", fontsize=8)
    axis.set_title(title, fontsize=8.5); axis.set_ylim(0, max(values)*1.25)
axes[0].set_ylabel("Percent")
fig.savefig(CFG["outdir"] / "fig3_decomposition.png"); plt.show()

# Figure 4: average predicted dissatisfaction probabilities from Equation (2)
prediction_rows = []
for mass_market in (0, 1):
    for disruption in (0, 1):
        scenario = df.assign(mass_market=mass_market, disruption=disruption)
        prediction_rows.append({"mass_market": mass_market, "disruption": disruption,
                                "probability_pct": 100 * m2_city.predict(scenario).mean()})
predictions = pd.DataFrame(prediction_rows)
fig, ax = plt.subplots(figsize=(4.5, 3.2))
for mass_market, color, label in [(0, C_SHORT, "Independent"), (1, C_LONG, "Intermediated")]:
    values = predictions.loc[predictions.mass_market.eq(mass_market), "probability_pct"].to_numpy()
    ax.plot([0, 1], values, "o-", color=color, lw=2, ms=7, label=label)
ax.set_xticks([0, 1], ["No disruption", "Reported disruption"])
ax.set_ylabel("Average predicted negative rating (%)"); ax.legend(frameon=False)
ax.set_title("Conditional customer impact", loc="left", fontweight="bold")
fig.savefig(CFG["outdir"] / "fig4_interaction.png"); plt.show()

# Figure 5: country estimates
country_plot = T7[~T7.omitted].sort_values("OR")
fig, ax = plt.subplots(figsize=(5.8, 3.5)); y = np.arange(len(country_plot))
ax.hlines(y, country_plot.CI_low, country_plot.CI_high, color=C_NEUT, lw=1.6)
ax.scatter(country_plot.OR, y, color=[C_LONG if value else "#AAAAAA" for value in country_plot.survives], zorder=3)
ax.axvline(1, color=C_NEUT, ls="--", lw=0.8); ax.set_xscale("log")
ax.set_yticks(y, [f"{country} (events={events})" for country, events in zip(country_plot.country, country_plot.events)])
ax.set_xlabel("Exposure odds ratio (log scale)")
ax.set_title("Exploratory country estimates", loc="left", fontweight="bold")
fig.savefig(CFG["outdir"] / "fig5_country_exploratory.png"); plt.show()

# Figure 6: measurement-opportunity pattern
fig, ax = plt.subplots(figsize=(4.8, 3.2))
ax.bar(T12.stage_breadth, T12.disruption_pct, color=C_LONG, width=0.65)
ax.set_xlabel("Narrated stage breadth"); ax.set_ylabel("Reviews reporting disruption (%)")
ax.set_title("Stage breadth and reported disruption", loc="left", fontweight="bold")
fig.savefig(CFG["outdir"] / "fig6_measurement_opportunity.png"); plt.show()


## 13. De-identified exports, manifest and reference checks


In [ ]:
keep = [
    "place_id", "country", "city", "segment", "mass_market", "rating", "negative",
    "n_words", "log_words", "review_age", "translated", "repeated_text",
    "stage_breadth", "disruption", "recovery",
] + list(STAGES) + list(CONSTRUCTS)
analytical_public = df[keep].copy()
analytical_public["place_id"] = "P" + pd.Series(pd.factorize(analytical_public.place_id)[0], index=analytical_public.index).astype(str).str.zfill(5)
analytical_public.to_csv(CFG["outdir"] / "reviews_analytical_deidentified.csv", index=False)

reference_files = {
    "T0_exclusion_log.csv": "T0_exclusion_log.csv",
    "T1_sample_composition.csv": "T1_sample_composition.csv",
    "T2_construct_prevalence.csv": "T2_construct_prevalence.csv",
    "T3_decomposition.csv": "T3_decomposition.csv",
    "T4_models.csv": "T4_models.csv",
    "T5_firth.csv": "T5_firth.csv",
    "T6_fdr_constructs.csv": "T6_fdr_constructs.csv",
    "T7_countries.csv": "T7_countries.csv",
    "T8_joint_heterogeneity.csv": "T8_joint_heterogeneity.csv",
    "T9_equivalence_tost.csv": "T9_equivalence_tost.csv",
    "T10_robustness.csv": "T10_robustness.csv",
    "T11_leave_one_out.csv": "T11_leave_one_out.csv",
    "T12_stage_breadth.csv": "T12_stage_breadth.csv",
}

def compare_reference(actual_path, reference_path):
    actual = pd.read_csv(actual_path)
    reference = pd.read_csv(reference_path)
    if list(actual.columns) != list(reference.columns) or len(actual) != len(reference):
        raise AssertionError(f"Structural mismatch: {actual_path.name}")
    for column in actual.columns:
        if pd.api.types.is_numeric_dtype(reference[column]):
            np.testing.assert_allclose(
                pd.to_numeric(actual[column], errors="coerce"),
                pd.to_numeric(reference[column], errors="coerce"),
                rtol=5e-4, atol=5e-6, equal_nan=True,
            )
        else:
            left = actual[column].fillna("<NA>").astype(str).tolist()
            right = reference[column].fillna("<NA>").astype(str).tolist()
            if left != right:
                raise AssertionError(f"Text mismatch in {actual_path.name}: {column}")

if CFG["reference_dir"].exists() and REFERENCE_RUN:
    for actual_name, reference_name in reference_files.items():
        compare_reference(CFG["outdir"] / actual_name, CFG["reference_dir"] / reference_name)
    print("Reference check passed for T0–T12.")
else:
    print("Reference comparison skipped (reference outputs absent or input corpus differs).")

manifest_outputs = {}
for path in sorted(CFG["outdir"].iterdir()):
    if path.is_file():
        manifest_outputs[path.name] = sha256_file(path)

manifest = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "duration_seconds": round(time.time() - STARTED, 3),
    "seed": CFG["seed"],
    "environment": {
        "python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
        "statsmodels": statsmodels.__version__, "scipy": scipy.__version__, "matplotlib": matplotlib.__version__,
    },
    "specification": {
        "controls": CTRL, "fixed_effects_main": CFG["fe_main"],
        "fixed_effects_sensitivity": CFG["fe_sensitivity"],
        "optimizer": CFG["optimizer"], "maxiter": CFG["maxiter"],
        "cluster": CFG["cluster"], "min_events_country": CFG["min_country_events"],
    },
    "analytical_base": {
        "reviews": len(df), "providers": df.place_id.nunique(), "cities": df.city.nunique(),
        "countries": df.country.nunique(), "disruptions": int(df.disruption.sum()),
    },
    "inputs": {CFG["raw_csv"].name: raw_sha256},
    "outputs": manifest_outputs,
    "measurement_scope": (
        "Individual human-coding records and validation metrics are not included in this public version; "
        "the notebook reports computational dictionary diagnostics and does not infer absent validation statistics."
    ),
}
(CFG["outdir"] / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest["analytical_base"], indent=2))


## 14. Interpretation boundary

The indicators quantify **narrated content in the retrieved corpus**, not objective incident incidence, contractual governance, observed recovery capacity or environmental performance. Collection stratum is assigned by the query protocol and is independent of the focal review text, but it is not a randomized treatment. All estimates are associational.

Individual human-coding records and their validation metrics are not part of this public computational release. If such records are deposited separately, they should be versioned, documented and linked without exposing reviewer identities or review text contrary to platform terms.
